# 01 — Data Pre-Processing

**Purpose:** Load the de-identified Qualtrics export, clean it, and save a tidy processed dataset ready for analysis and summary tables.

**Input:** `data/raw/anonymised/dental_trauma_survey_responses.csv`  
**Output:** `data/processed/survey_clean.csv`

### Survey structure overview
- **Self-assessment:** Knowledge/confidence ratings (0–10) before seeing cases
- **Case 1 — Primary tooth avulsion:** Q1, Q2, Q3
- **Case 2 — Permanent tooth avulsion:** Q4, Q39, Q41, Q40, Q42
- **Case 3 — Crown fracture:** Q44, Q45, Q46, Q47
- **Score:** `SC0` — total correct out of 12
- **Experimental group:** derived from consent/prompt columns Q55 (No Resource), Q56 (PDF), Q54 (ChatGPT)

### Key output columns for analysis
| Column | Description |
|---|---|
| `group_label` | Experimental group (ordered categorical) |
| `edu_status` | Education level, normalised (ordered categorical) |
| `specialty` | Clinical specialty |
| `training_level` | Year/PGY/years-since-residency |
| `duration_min` | Survey completion time in minutes |
| `self_knowledge_tdi` | Self-rated TDI knowledge (0–10) |
| `self_confidence_avulsion` | Self-rated avulsion confidence (0–10) |
| `self_confidence_fracture` | Self-rated fracture confidence (0–10) |
| `self_confidence_mean` | Mean of the three self-ratings (0–10) |
| `n_correct` | Questions answered correctly (0–12) |
| `n_incorrect` | Questions answered incorrectly |
| `n_not_sure` | Questions answered "I'm not sure" |
| `pct_correct_of_attempted` | % correct among attempted questions (excludes "not sure") |
| `c1_*` … `c3_*` | Raw response text per question |
| `c1_*_correct` … `c3_*_correct` | Binary correct/incorrect per question |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load raw data

Qualtrics exports have **3 header rows**:
- Row 0: short variable names (used as column headers)
- Row 1: full question text
- Row 2: Qualtrics import IDs

We keep row 0 as the header and drop rows 1–2.

In [2]:
RAW_PATH = Path("../data/raw/anonymised/dental_trauma_survey_responses.csv")

df_raw = pd.read_csv(RAW_PATH, header=0, skiprows=[1, 2], low_memory=False)

print(f"Shape: {df_raw.shape}")
df_raw.head(3)

Shape: (21, 33)


,Progress,Duration (in seconds),Finished,Q1_1,Q2_1,Q3_1,Q5,Q5_4_TEXT,Medical Student,Resident/Fellow,...,Q41,Q40,Q42,Q44,Q45,Q46,Q47,SC0,block2progress,random
0,100,451,True,3,5,4,Attending,NaN,NaN,NaN,...,False,Saline>Milk>Tap Water>Air,Antibiotics are not indicated,Complicated crown fracture of a primary incisor,The patient should be sent to a dentist as soo...,Chest and/or soft tissues of the face,All of these are indications for antibiotic th...,7.0,NaN,2.0
1,100,683,True,1,6,6,Medical Student,NaN,M4,NaN,...,False,Milk>Saline>Tap Water>Air,Doxycycline BID for 7 days,Complicated crown fracture of a permanent incisor,The patient should be sent to a dentist as soo...,Chest,All of these are indications for antibiotic th...,8.0,NaN,2.0
2,100,178,True,5,7,6,Resident/Fellow,NaN,NaN,Other (please specify),...,False,Milk>Saline>Tap Water>Air,Antibiotics are not indicated,Uncomplicated crown fracture of a permanent in...,The patient should be sent to a dentist as soo...,Chest,All of these are indications for antibiotic th...,7.0,NaN,2.0


## 2. Filter to complete responses

Keep only rows where `Finished == True` and `Progress == 100`.

In [3]:
df = df_raw.copy()

# Qualtrics exports Finished as a boolean or the string "True"/"False" depending on
# the export format. Normalise to bool so the filter works in both cases.
finished_bool = df["Finished"].map(lambda x: str(x).strip().lower() == "true")

print(f"Total responses before filtering: {len(df)}")
print(f"  Finished=True:   {finished_bool.sum()}")
print(f"  Finished=False:  {(~finished_bool).sum()}")

df = df[finished_bool & df["Progress"].astype(int).eq(100)].copy()
df = df.reset_index(drop=True)

print(f"\nResponses after filtering: {len(df)}")

Total responses before filtering: 21
  Finished=True:   18
  Finished=False:  3

Responses after filtering: 18


## 3. Drop metadata / PII columns

In [4]:
drop_cols = [
    # Already removed by anonymise_raw_survey.py, listed here defensively
    "StartDate", "EndDate", "Status", "IPAddress",
    "RecordedDate", "ResponseId",
    "RecipientLastName", "RecipientFirstName", "RecipientEmail",
    "ExternalReference", "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage",
    # Progress/Finished used for filtering above; no longer needed
    "Progress", "Finished",
    # NOTE: Q55/Q56/Q54 are kept here — they encode the experimental group
    # (Q55 = No Resource, Q56 = PDF, Q54 = ChatGPT) and are parsed in Section 5.
    "block2progress",
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['Duration (in seconds)', 'Q1_1', 'Q2_1', 'Q3_1', 'Q5', 'Q5_4_TEXT', 'Medical Student', 'Resident/Fellow', 'Resident/Fellow_4_TEXT', 'Resident/Fellow.1', 'Resident/Fellow_5_TEXT', 'Attending', 'Attending.1', 'Q55', 'Q56', 'Q54', 'Q1', 'Q2', 'Q3', 'Q4', 'Q39', 'Q41', 'Q40', 'Q42', 'Q44', 'Q45', 'Q46', 'Q47', 'SC0', 'random']


## 4. Rename columns to readable names

In [5]:
rename_map = {
    "ResponseId":               "response_id",
    "Duration (in seconds)":    "duration_sec",
    # Self-assessment
    "Q1_1":  "self_knowledge_tdi",
    "Q2_1":  "self_confidence_avulsion",
    "Q3_1":  "self_confidence_fracture",
    # Demographics
    "Q5":                    "edu_status",
    "Q5_4_TEXT":             "edu_status_other",
    "Medical Student":       "ms_year",
    "Resident/Fellow":       "res_specialty",
    "Resident/Fellow_4_TEXT": "res_specialty_other",
    "Resident/Fellow_5_TEXT": "res_pgy_other",    # second Resident/Fellow col handled below
    "Attending":             "att_specialty",      # first Attending col
    # second Attending col (years since residency) handled below
    # Knowledge questions — Case 1 (primary avulsion)
    "Q1":  "c1_injury_type",
    "Q2":  "c1_treatment",
    "Q3":  "c1_antibiotics",
    # Knowledge questions — Case 2 (permanent avulsion)
    "Q4":  "c2_injury_type",
    "Q39": "c2_treatment",
    "Q41": "c2_tf_60min",
    "Q40": "c2_storage_rank",
    "Q42": "c2_antibiotics",
    # Knowledge questions — Case 3 (crown fracture)
    "Q44": "c3_injury_type",
    "Q45": "c3_treatment",
    "Q46": "c3_imaging",
    "Q47": "c3_antibiotics",
    # Outcome
    "SC0":    "total_score",
    # 'random' is a Qualtrics randomisation flag (values 1/2) that does not
    # reliably map to the three experimental groups. Kept for diagnostics only.
    "random": "random_assignment",
}

# Handle duplicate column names that Qualtrics creates for branched questions
cols = df.columns.tolist()
res_fellow_indices  = [i for i, c in enumerate(cols) if c == "Resident/Fellow"]
attending_indices   = [i for i, c in enumerate(cols) if c == "Attending"]

if len(res_fellow_indices) == 2:
    cols[res_fellow_indices[1]] = "res_pgy"
if len(attending_indices) == 2:
    cols[attending_indices[1]] = "att_years_since_residency"

df.columns = cols
df = df.rename(columns=rename_map)

print(df.columns.tolist())

['duration_sec', 'self_knowledge_tdi', 'self_confidence_avulsion', 'self_confidence_fracture', 'edu_status', 'edu_status_other', 'ms_year', 'res_specialty', 'res_specialty_other', 'Resident/Fellow.1', 'res_pgy_other', 'att_specialty', 'Attending.1', 'Q55', 'Q56', 'Q54', 'c1_injury_type', 'c1_treatment', 'c1_antibiotics', 'c2_injury_type', 'c2_treatment', 'c2_tf_60min', 'c2_storage_rank', 'c2_antibiotics', 'c3_injury_type', 'c3_treatment', 'c3_imaging', 'c3_antibiotics', 'total_score', 'random_assignment']


## 5. Derive experimental group

Group assignment is encoded in three separate consent/prompt columns:

| Column | Value present | Group |
|---|---|---|
| Q55 | "I have read the above statement on material use" | No Resource |
| Q56 | "I have read the above statement and opened the PDF window" | PDF |
| Q54 | "I have read the above statement and opened the ChatGPT window" | ChatGPT |

The `random_assignment` column (Qualtrics internal, values 1/2) is retained for diagnostics
but does **not** reliably identify the three groups and should not be used for analysis.

In [6]:
def derive_group(row):
    """Return the experimental group label based on which prompt column is filled."""
    if pd.notna(row.get("Q56")) and str(row["Q56"]).strip():
        return "PDF"
    if pd.notna(row.get("Q54")) and str(row["Q54"]).strip():
        return "ChatGPT"
    if pd.notna(row.get("Q55")) and str(row["Q55"]).strip():
        return "No Resource"
    return np.nan


df["group_label"] = df.apply(derive_group, axis=1)

# Drop the raw prompt columns now that group is captured
df = df.drop(columns=["Q55", "Q56", "Q54"])

print("Group distribution:")
print(df["group_label"].value_counts(dropna=False))
print()
print("Cross-check against random_assignment (should NOT be 1:1):")
print(pd.crosstab(df["group_label"], df["random_assignment"], dropna=False))

Group distribution:
group_label
No Resource    7
ChatGPT        6
PDF            5
Name: count, dtype: int64

Cross-check against random_assignment (should NOT be 1:1):
random_assignment  1.0  2.0
group_label                
ChatGPT              3    3
No Resource          3    4
PDF                  3    2


## 5. Fix data types

In [7]:
df["duration_sec"]       = pd.to_numeric(df["duration_sec"], errors="coerce")
df["total_score"]        = pd.to_numeric(df["total_score"], errors="coerce")
df["random_assignment"]  = pd.to_numeric(df["random_assignment"], errors="coerce").astype("Int64")

self_rating_cols = ["self_knowledge_tdi", "self_confidence_avulsion", "self_confidence_fracture"]
df[self_rating_cols] = df[self_rating_cols].apply(pd.to_numeric, errors="coerce")

# Duration in minutes (more interpretable than seconds)
df["duration_min"] = (df["duration_sec"] / 60).round(2)

# Mean of the three self-assessment ratings — used for confidence vs score correlation
df["self_confidence_mean"] = df[self_rating_cols].mean(axis=1).round(2)

df[["duration_sec", "duration_min", "self_confidence_mean"]].head()

,duration_sec,duration_min,self_confidence_mean
0,451,7.52,4.00
1,683,11.38,4.33
2,178,2.97,6.00
3,885,14.75,3.00
4,540,9.00,1.67


## 6. Consolidate demographic columns

Qualtrics branches demographic questions by role, so specialty and training level are spread across multiple columns. We unify them into single columns.

In [8]:
def consolidate_specialty(row):
    """Return a single specialty string regardless of respondent role."""
    if pd.notna(row.get("att_specialty")) and str(row["att_specialty"]).strip():
        return str(row["att_specialty"]).strip()
    if pd.notna(row.get("res_specialty")) and str(row["res_specialty"]).strip():
        spec = str(row["res_specialty"]).strip()
        if spec == "Other (please specify)" and pd.notna(row.get("res_specialty_other")):
            return str(row["res_specialty_other"]).strip()
        return spec
    return np.nan


def consolidate_training_level(row):
    """Return a unified training / experience level string."""
    status = str(row.get("edu_status", "")).strip()
    if status == "Medical Student":
        return str(row.get("ms_year", "")).strip() or np.nan
    if status == "Resident/Fellow":
        pgy = str(row.get("res_pgy", "")).strip()
        if pgy == "Other (please specify)" and pd.notna(row.get("res_pgy_other")):
            return str(row["res_pgy_other"]).strip()
        return pgy or np.nan
    if status == "Attending":
        return str(row.get("att_years_since_residency", "")).strip() or np.nan
    return np.nan


df["specialty"]      = df.apply(consolidate_specialty, axis=1)
df["training_level"] = df.apply(consolidate_training_level, axis=1)

# Resolve free-text for 'Other' edu_status entries
df["edu_status"] = df.apply(
    lambda r: str(r["edu_status_other"]).strip()
    if str(r.get("edu_status", "")).strip() == "Other (please specify)"
    and pd.notna(r.get("edu_status_other"))
    else str(r["edu_status"]).strip(),
    axis=1,
)

# Normalise advanced practice provider variants to a single label
APP_VARIANTS = {"pnp", "np", "app", "pediatric nurse practitioner", "arnp", "crna", "pa"}
df["edu_status"] = df["edu_status"].apply(
    lambda v: "Advanced Practice Provider"
    if str(v).strip().lower() in APP_VARIANTS
    else v
)

# Drop the now-redundant branched columns
branch_cols = [
    "edu_status_other", "ms_year",
    "res_specialty", "res_specialty_other", "res_pgy", "res_pgy_other",
    "att_specialty", "att_years_since_residency",
]
df = df.drop(columns=[c for c in branch_cols if c in df.columns])

print("edu_status values:")
print(df["edu_status"].value_counts(dropna=False))
df[["edu_status", "specialty", "training_level"]].head(10)

edu_status values:
edu_status
Attending                     6
Resident/Fellow               5
Medical Student               4
Advanced Practice Provider    3
Name: count, dtype: int64


,edu_status,specialty,training_level
0,Attending,Emergency Medicine,NaN
1,Medical Student,NaN,M4
2,Resident/Fellow,Pediatric Emergency Medicine,NaN
3,Medical Student,NaN,M4
4,Medical Student,NaN,M4
5,Resident/Fellow,Family Medicine,NaN
6,Medical Student,NaN,M4
7,Attending,Emergency Medicine,NaN
8,Resident/Fellow,Emergency Medicine,NaN
9,Attending,Emergency Medicine,NaN


## 7. Encode ordered categoricals

Ordered categoricals ensure correct sort order in plots and summary tables without manual sorting.

In [9]:
GROUP_ORDER = ["No Resource", "PDF", "ChatGPT"]
EDU_ORDER   = ["Medical Student", "Resident/Fellow", "Attending", "Advanced Practice Provider"]

df["group_label"] = pd.Categorical(df["group_label"], categories=GROUP_ORDER, ordered=True)
df["edu_status"]  = pd.Categorical(df["edu_status"],  categories=EDU_ORDER,   ordered=True)

print("group_label categories:", df["group_label"].cat.categories.tolist())
print("edu_status  categories:", df["edu_status"].cat.categories.tolist())
print()
print(pd.crosstab(df["edu_status"], df["group_label"], margins=True))

group_label categories: ['No Resource', 'PDF', 'ChatGPT']
edu_status  categories: ['Medical Student', 'Resident/Fellow', 'Attending', 'Advanced Practice Provider']

group_label                 No Resource  PDF  ChatGPT  All
edu_status                                                
Medical Student                       1    1        2    4
Resident/Fellow                       3    1        1    5
Attending                             3    2        1    6
Advanced Practice Provider            0    1        2    3
All                                   7    5        6   18


## 8. Score individual questions (correct / incorrect)

The answer key is loaded directly from `data/raw/anonymised/answers_to_the_questions_form_questions.csv`
so this notebook stays in sync with the canonical answers file. Each response is encoded as
1 (correct) or 0 (incorrect), with NaN for missing responses.

In [10]:
ANSWERS_PATH = Path("../data/raw/anonymised/answers_to_the_questions_form_questions.csv")

answers_df = pd.read_csv(ANSWERS_PATH)

# Map Q-codes → internal column names
QCODE_TO_COL = {
    "Q1":  "c1_injury_type",
    "Q2":  "c1_treatment",
    "Q3":  "c1_antibiotics",
    "Q4":  "c2_injury_type",
    "Q39": "c2_treatment",
    "Q41": "c2_tf_60min",
    "Q40": "c2_storage_rank",
    "Q42": "c2_antibiotics",
    "Q44": "c3_injury_type",
    "Q45": "c3_treatment",
    "Q46": "c3_imaging",
    "Q47": "c3_antibiotics",
}

# Human-readable short labels — used for axis labels and titles in all analysis plots
QUESTION_LABELS = {
    "c1_injury_type":  "C1: Injury type",
    "c1_treatment":    "C1: Treatment",
    "c1_antibiotics":  "C1: Antibiotics",
    "c2_injury_type":  "C2: Injury type",
    "c2_treatment":    "C2: Treatment",
    "c2_tf_60min":     "C2: >60 min outside mouth (T/F)",
    "c2_storage_rank": "C2: Storage medium ranking",
    "c2_antibiotics":  "C2: Antibiotic therapy",
    "c3_injury_type":  "C3: Injury type",
    "c3_treatment":    "C3: Treatment",
    "c3_imaging":      "C3: Imaging area",
    "c3_antibiotics":  "C3: Antibiotic indication",
}

ANSWER_KEY = {
    QCODE_TO_COL[row["question_col"]]: row["correct_answer"]
    for _, row in answers_df.iterrows()
}


def score_answer(response, correct):
    """Return 1 if response matches the correct answer, 0 if not, NaN if missing."""
    if pd.isna(response):
        return np.nan
    return int(str(response).strip().lower() == str(correct).strip().lower())


for col, answer in ANSWER_KEY.items():
    df[f"{col}_correct"] = df[col].apply(lambda r: score_answer(r, answer))

correct_cols = [c for c in df.columns if c.endswith("_correct")]
df[correct_cols].describe()

,c1_injury_type_correct,c1_treatment_correct,c1_antibiotics_correct,c2_injury_type_correct,c2_treatment_correct,c2_tf_60min_correct,c2_storage_rank_correct,c2_antibiotics_correct,c3_injury_type_correct,c3_treatment_correct,c3_imaging_correct,c3_antibiotics_correct
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.944444,0.944444,0.888889,0.833333,0.722222,0.888889,0.833333,0.055556,0.666667,0.888889,0.444444,0.111111
std,0.235702,0.235702,0.323381,0.383482,0.460889,0.323381,0.383482,0.235702,0.485071,0.323381,0.511310,0.323381
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,1.000000,0.250000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 9. Summarise correct / incorrect / not sure per respondent

In [11]:
question_cols = list(ANSWER_KEY.keys())
correct_cols  = [f"{c}_correct" for c in question_cols]

# n_correct / n_incorrect from binary scored columns (NaN = skipped, not counted)
df["n_correct"]   = df[correct_cols].sum(axis=1, skipna=True).astype("Int64")
df["n_incorrect"] = df[correct_cols].eq(0).sum(axis=1).astype("Int64")

# "I'm not sure" is a literal survey response option
df["n_not_sure"] = (
    df[question_cols]
    .apply(lambda col: col.str.strip().str.lower().eq("i'm not sure"))
    .sum(axis=1)
    .astype("Int64")
)

# Proportion correct among attempted questions only (excludes "not sure" from denominator)
attempted = df["n_correct"] + df["n_incorrect"]
df["pct_correct_of_attempted"] = ((df["n_correct"] / attempted) * 100).round(1)
df[["n_correct", "n_incorrect", "n_not_sure", "pct_correct_of_attempted"]].describe()

,n_correct,n_incorrect,n_not_sure,pct_correct_of_attempted
count,18.0,18.0,18.0,18.0
mean,8.222222,3.777778,0.444444,68.522222
std,1.832888,1.832888,1.149026,15.270408
min,5.0,1.0,0.0,41.7
25%,7.0,2.25,0.0,58.3
50%,8.0,4.0,0.0,66.7
75%,9.75,5.0,0.0,81.225
max,11.0,7.0,4.0,91.7


## 10. Final overview

In [12]:
print(f"Final dataset shape: {df.shape}")

print("\n--- Group distribution ---")
print(df["group_label"].value_counts().to_string())

print("\n--- Education level × Group (crosstab) ---")
print(pd.crosstab(df["edu_status"], df["group_label"], margins=True).to_string())

print("\n--- Score summary by group ---")
score_cols = ["n_correct", "n_incorrect", "n_not_sure", "pct_correct_of_attempted"]
print(df.groupby("group_label", observed=True)[score_cols].describe().round(2).to_string())

print("\n--- Duration summary by group (minutes) ---")
print(df.groupby("group_label", observed=True)["duration_min"].describe().round(2).to_string())

print("\n--- Self-confidence summary ---")
conf_cols = ["self_knowledge_tdi", "self_confidence_avulsion", "self_confidence_fracture", "self_confidence_mean"]
print(df[conf_cols].describe().round(2).to_string())

print("\n--- Missing values ---")
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing.to_string() if not missing.empty else "None")

df[["group_label", "edu_status", "specialty", "duration_min",
    "self_confidence_mean", "n_correct", "n_incorrect",
    "n_not_sure", "pct_correct_of_attempted"]].head(10)

Final dataset shape: (18, 42)

--- Group distribution ---
group_label
No Resource    7
ChatGPT        6
PDF            5

--- Education level × Group (crosstab) ---
group_label                 No Resource  PDF  ChatGPT  All
edu_status                                                
Medical Student                       1    1        2    4
Resident/Fellow                       3    1        1    5
Attending                             3    2        1    6
Advanced Practice Provider            0    1        2    3
All                                   7    5        6   18

--- Score summary by group ---


            n_correct                                         n_incorrect                                       n_not_sure                                      pct_correct_of_attempted                                              
                count  mean   std  min   25%  50%   75%   max       count  mean   std  min  25%  50%   75%  max      count  mean   std  min  25%  50%  75%  max                    count   mean    std   min   25%    50%    75%   max
group_label                                                                                                                                                                                                                           
No Resource       7.0  7.57   1.9  5.0   6.5  8.0   8.5  10.0         7.0  4.43   1.9  2.0  3.5  4.0   5.5  7.0        7.0   1.0  1.73  0.0  0.0  0.0  1.5  4.0                      7.0  63.11  15.83  41.7  54.2   66.7  70.85  83.3
PDF               5.0   8.4  1.82  6.0   7.0  9.0  10.0  10.0         5.0   

,group_label,edu_status,specialty,duration_min,self_confidence_mean,n_correct,n_incorrect,n_not_sure,pct_correct_of_attempted
0,No Resource,Attending,Emergency Medicine,7.52,4.00,8,4,0,66.7
1,ChatGPT,Medical Student,NaN,11.38,4.33,9,3,0,75.0
2,No Resource,Resident/Fellow,Pediatric Emergency Medicine,2.97,6.00,8,4,0,66.7
3,PDF,Medical Student,NaN,14.75,3.00,10,2,0,83.3
4,ChatGPT,Medical Student,NaN,9.00,1.67,7,5,0,58.3
5,No Resource,Resident/Fellow,Family Medicine,5.92,0.67,5,7,3,41.7
6,No Resource,Medical Student,NaN,4.42,0.00,5,7,4,41.7
7,PDF,Attending,Emergency Medicine,88.68,8.33,6,6,0,50.0
8,ChatGPT,Resident/Fellow,Emergency Medicine,6.37,6.33,7,5,1,58.3
9,ChatGPT,Attending,Emergency Medicine,14.52,6.33,11,1,0,91.7


## 11. Save processed data

In [13]:
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Main processed dataset
survey_path = OUT_DIR / "survey_clean.csv"
df.to_csv(survey_path, index=False)
print(f"Saved {len(df)} rows x {df.shape[1]} columns to {survey_path}")

# Question labels map — loaded by analysis notebooks for axis labels and titles
labels_path = OUT_DIR / "question_labels.csv"
pd.DataFrame(
    list(QUESTION_LABELS.items()), columns=["column", "label"]
).to_csv(labels_path, index=False)
print(f"Saved question labels to {labels_path}")

print("\nColumns in survey_clean.csv:")
print(df.columns.tolist())

Saved 18 rows x 42 columns to ..\data\processed\survey_clean.csv
Saved question labels to ..\data\processed\question_labels.csv

Columns in survey_clean.csv:
['duration_sec', 'self_knowledge_tdi', 'self_confidence_avulsion', 'self_confidence_fracture', 'edu_status', 'Resident/Fellow.1', 'Attending.1', 'c1_injury_type', 'c1_treatment', 'c1_antibiotics', 'c2_injury_type', 'c2_treatment', 'c2_tf_60min', 'c2_storage_rank', 'c2_antibiotics', 'c3_injury_type', 'c3_treatment', 'c3_imaging', 'c3_antibiotics', 'total_score', 'random_assignment', 'group_label', 'duration_min', 'self_confidence_mean', 'specialty', 'training_level', 'c1_injury_type_correct', 'c1_treatment_correct', 'c1_antibiotics_correct', 'c2_injury_type_correct', 'c2_treatment_correct', 'c2_tf_60min_correct', 'c2_storage_rank_correct', 'c2_antibiotics_correct', 'c3_injury_type_correct', 'c3_treatment_correct', 'c3_imaging_correct', 'c3_antibiotics_correct', 'n_correct', 'n_incorrect', 'n_not_sure', 'pct_correct_of_attempted']
